In [0]:
crm_df = spark.table("workspace.default.crm_silver")
billing_df = spark.table("workspace.default.billing_silver")
analytics_df = spark.table("workspace.default.analytics_silver")

In [0]:
import builtins
from pyspark.sql.functions import col, when, sum

crm_missing = crm_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in crm_df.columns
]).collect()[0]

billing_missing = billing_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in billing_df.columns
]).collect()[0]

analytics_missing = analytics_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in analytics_df.columns
]).collect()[0]

# Use Python's built-in sum(), not Spark's sum()
total_missing = (
    builtins.sum(crm_missing.asDict().values()) +
    builtins.sum(billing_missing.asDict().values()) +
    builtins.sum(analytics_missing.asDict().values())
)

print("Total Missing Values:", total_missing)

Total Missing Values: 0


In [0]:
crm_records = crm_df.count()
billing_records = billing_df.count()
analytics_records = analytics_df.count()

total_records = (
    crm_records +
    billing_records +
    analytics_records
)

print("CRM Records:", crm_records)
print("Billing Records:", billing_records)
print("Analytics Records:", analytics_records)
print("Total Records:", total_records)


CRM Records: 10195
Billing Records: 11454
Analytics Records: 894
Total Records: 22543


In [0]:
missing_percentage = (total_missing / total_records) * 100

print("Missing Percentage:", round(missing_percentage, 2), "%")

Missing Percentage: 0.0 %


In [0]:
volume_difference = abs(crm_records - billing_records)
volume_drift = (volume_difference / crm_records) * 100

trust_score = 100 - missing_percentage - volume_drift

if trust_score < 0:
    trust_score = 0

print("Volume Drift:", round(volume_drift, 2), "%")
print("Trust Score:", round(trust_score, 2))

Volume Drift: 12.35 %
Trust Score: 87.65


In [0]:
if trust_score >= 90:
    print("🟢 Excellent Data Quality")

elif trust_score >= 75:
    print("🟡 Good Data Quality")

elif trust_score >= 60:
    print("🟠 Average Data Quality")

else:
    print("🔴 Poor Data Quality")

🟡 Good Data Quality


In [0]:
print("=" * 50)
print("      DATA TRUST SCORE REPORT")
print("=" * 50)

print(f"CRM Records        : {crm_records}")
print(f"Billing Records    : {billing_records}")
print(f"Analytics Records  : {analytics_records}")

print(f"\nTotal Missing      : {total_missing}")
print(f"Missing Percentage : {round(missing_percentage,2)} %")

print(f"Volume Drift       : {round(volume_drift,2)} %")

print(f"\nOverall Trust Score: {round(trust_score,2)} / 100")

if trust_score >= 90:
    status = "Excellent"
elif trust_score >= 75:
    status = "Good"
elif trust_score >= 60:
    status = "Average"
else:
    status = "Poor"

print(f"Trust Level        : {status}")

print("=" * 50)

      DATA TRUST SCORE REPORT
CRM Records        : 10195
Billing Records    : 11454
Analytics Records  : 894

Total Missing      : 0
Missing Percentage : 0.0 %
Volume Drift       : 12.35 %

Overall Trust Score: 87.65 / 100
Trust Level        : Good


In [0]:
from pyspark.sql import Row

trust_df = spark.createDataFrame([

    Row(
        crm_records=crm_records,
        billing_records=billing_records,
        analytics_records=analytics_records,

        total_missing=total_missing,
        missing_percentage=round(missing_percentage,2),

        volume_drift=round(volume_drift,2),

        trust_score=round(trust_score,2),

        trust_level=status
    )

])

display(trust_df)

crm_records,billing_records,analytics_records,total_missing,missing_percentage,volume_drift,trust_score,trust_level
10195,11454,894,0,0.0,12.35,87.65,Good


In [0]:
trust_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.trust_score_gold")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM workspace.default.trust_score_gold
    """)
)

crm_records,billing_records,analytics_records,total_missing,missing_percentage,volume_drift,trust_score,trust_level
10195,11454,894,0,0.0,12.35,87.65,Good


In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY workspace.default.trust_score_gold
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-07-11T20:43:50.000Z,72969947334986,khandelwalvishwa1310@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1352019682178603),7bda1d37-6170-4f64-9200-b24fe39dd5eb,0711-201517-flo0f3me-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1, numOutputBytes -> 2505)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
